# Biological Sequence Motif Discovery
## B.Tech Major Project — IIEST Shibpur
**Presented by:** Ramesh Chandra Soren (2022CSB086)  
**Supervisor:** Dr. Surajeet Ghosh  
**Date:** 17 March, 2026

---
This notebook provides an interactive walkthrough of the motif discovery pipeline:
1. Data loading & preprocessing
2. MEME algorithm
3. Gibbs Sampling
4. Evaluation & visualisation

In [ ]:
import sys, os
sys.path.insert(0, '../src')

from preprocess     import read_fasta, filter_sequences, compute_background_frequencies
from meme_algorithm import MEME
from gibbs_sampling import GibbsMotifFinder
from evaluate       import site_metrics, read_true_positions_from_fasta
from visualize      import (plot_sequence_logo, plot_pwm_heatmap,
                             plot_gibbs_convergence, plot_algorithm_comparison,
                             plot_site_distribution)

%matplotlib inline
import matplotlib.pyplot as plt
print('Imports OK')

## 1. Load and Preprocess Data

In [ ]:
# ── Choose dataset ──────────────────────────────────────────────────────────
FASTA   = '../data/raw/synthetic_tata_benchmark.fasta'
WIDTH   = 8
TRUE_MOTIF = 'TATATAAG'

raw = read_fasta(FASTA)
filtered, stats = filter_sequences(raw, min_length=WIDTH + 10)
sequences = [s for _, s in filtered]
bg        = compute_background_frequencies(filtered)
true_pos  = read_true_positions_from_fasta(FASTA)

print(f'Sequences loaded : {len(sequences)}')
print(f'Background freqs : {bg}')
print(f'True motif       : {TRUE_MOTIF}')
print(f'True positions   : {list(true_pos.items())[:5]} ...')

## 2. Run MEME Algorithm

In [ ]:
meme = MEME(sequences, width=WIDTH, n_motifs=2, model='ZOOPS',
            n_seeds=8, max_iter=30, background=bg)
meme_motifs = meme.fit(verbose=True)
meme.summary()

In [ ]:
# Sequence logo for best MEME motif
best = meme_motifs[0]
plot_sequence_logo(best['pwm'],
                   title=f"MEME Motif 1 — {best['consensus']}")
plt.show()

In [ ]:
# PWM heatmap
plot_pwm_heatmap(best['pwm'], title='MEME PWM Heatmap')
plt.show()

## 3. Run Gibbs Sampling

In [ ]:
gibbs = GibbsMotifFinder(sequences, width=WIDTH, n_runs=5,
                          n_iter=200, background=bg)
gibbs_pwm, gibbs_score = gibbs.fit(verbose=True)
gibbs.summary()

In [ ]:
plot_sequence_logo(gibbs_pwm, title=f"Gibbs Motif — {gibbs_pwm.consensus()}")
plt.show()

In [ ]:
# Convergence across runs
plot_gibbs_convergence(gibbs.run_histories,
                        labels=[f'Run {i+1}' for i in range(len(gibbs.run_histories))],
                        title='Gibbs Sampling Convergence')
plt.show()

## 4. Evaluation

In [ ]:
# Assign seq_index to MEME sites
meme_sites = meme_motifs[0]['sites']
for i, s in enumerate(meme_sites):
    s.setdefault('seq_index', i)

meme_eval  = site_metrics(meme_sites, true_pos, WIDTH)
gibbs_eval = site_metrics(gibbs.best_sites, true_pos, WIDTH)

print(f"{'Metric':<20} {'MEME':>10} {'Gibbs':>10}")
print('─' * 42)
for k in ['sensitivity','ppv','f1','nCC','site_accuracy']:
    print(f"{k:<20} {meme_eval[k]:>10.4f} {gibbs_eval[k]:>10.4f}")

## 5. Try Your Own Sequences

In [ ]:
# ── Paste your own FASTA path here ──────────────────────────────────────────
custom_fasta = '../data/raw/ecoli_sigma70_promoters.fasta'
custom_width = 6

raw2 = read_fasta(custom_fasta)
filtered2, _ = filter_sequences(raw2, min_length=custom_width + 5)
seqs2 = [s for _, s in filtered2]
bg2   = compute_background_frequencies(filtered2)
print(f'Loaded {len(seqs2)} sequences')

m2 = MEME(seqs2, width=custom_width, n_motifs=1, model='ZOOPS',
           n_seeds=8, max_iter=25, background=bg2)
m2_motifs = m2.fit(verbose=True)
if m2_motifs:
    plot_sequence_logo(m2_motifs[0]['pwm'],
                       title=f"Custom FASTA — {m2_motifs[0]['consensus']}")
    plt.show()